In [1]:
import os
import pandas as pd
import numpy as np
import boto3
import time
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-07 21:30:57.343284


### Constants

In [3]:
str_project = '20231010-gen-xii'
str_dirname_output = './output'
#str_variant = 'model3'

### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import data

In [5]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/08_retro_scoring/06_create_df/{str_filename}'
df = pd.read_parquet(str_uri)
# replace
df.replace(['NaN','nan'], np.nan, inplace=True)

# show
df

,uniqueid__app_x,bigstatusid__app,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,approvaldate__app,bitfunded__app,fundeddate__app,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
0,0__0__20210120,14.0,Stanley,Virginia,22851,False,0.0,NaN,False,NaN,...,0.0,0.00,412.34,0.307483,0.103700,0,1.149789,suv,1,2011-06-30 10:15:42.320
1,0__0__20210120,14.0,Austin,Texas,78727,False,0.0,NaN,False,NaN,...,250.0,105.06,638.00,0.233488,0.132294,0,1.100576,suv,0,2012-10-19 15:20:30.163
2,0__0__20210122,14.0,STOCKTON,California,95207,False,0.0,NaN,False,NaN,...,1000.0,1000.00,629.99,0.340494,0.140493,0,1.051327,suv,1,2020-07-06 17:09:57.383
3,0__0__20210123,14.0,Mount Orab,Ohio,45154,False,0.0,NaN,False,NaN,...,0.0,0.00,398.95,0.481666,0.070066,0,1.143705,auto,0,2016-11-16 16:28:03.170
4,0__0__20210123,14.0,Mount Orab,Ohio,45154,False,0.0,NaN,False,NaN,...,0.0,0.00,398.95,0.481666,0.070066,0,1.143705,auto,0,2016-11-16 16:28:03.170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71184,0__0__20231109,5.0,FOREST HILL,Texas,76119,False,0.0,NaN,False,NaN,...,0.0,0.00,511.77,0.449811,0.122139,0,1.349948,suv,0,2008-02-20 11:14:02.013
71185,0__0__20231120,5.0,CHICAGO,Illinois,60629,False,0.0,NaN,False,NaN,...,1000.0,500.00,450.63,0.350022,0.150022,0,1.149932,suv,0,2013-05-08 08:47:59.997
71186,0__0__20231107,1.0,PIKESVILLE,Maryland,21208,False,0.0,NaN,False,NaN,...,500.0,1000.00,727.74,0.385896,0.058485,1,0.838573,suv,0,2023-05-03 15:53:03.583
71187,0__0__20231117,5.0,Saginaw,Michigan,48601,False,0.0,NaN,False,NaN,...,0.0,0.00,587.21,0.453596,0.128596,0,1.121384,suv,1,2017-07-26 16:22:04.930


### Date range

In [6]:
dtm_min = df['dtmFunded'].min()
dtm_max = df['dtmFunded'].max()
print(f'Min date: {dtm_min}')
print(f'Max date: {dtm_max}')

Min date: 2021-01-25 00:00:00
Max date: 2024-03-04 00:00:00


### Get the features in the models

In [7]:
# list_cols = []
# for str_model in ['01_ad','02_pricing_pd','03_pricing_lgd']:
#     str_filename = 'df_cols_in_model.csv'
#     str_uri = f's3://{str_project}/{str_model}/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
#     list_cols_model = list(pd.read_csv(str_uri)['feature'])
#     list_cols += list_cols_model

# # rm dups
# list_cols = list(dict.fromkeys(list_cols))
# print(f'There are {len(list_cols)} features in all models')

### Application

In [8]:
# app
list_cols_raw_app_suffix = [col for col in df.columns if '__app' in col]
print(f'There are {len(list_cols_raw_app_suffix)} app features')
for a, col in enumerate(list_cols_raw_app_suffix):
    print(f'{a+1} - {col}')

There are 93 app features
1 - uniqueid__app_x
2 - bigstatusid__app
3 - strcity__app
4 - strname__app
5 - strzipcode__app
6 - bitapproved__app
7 - bitsystemdecline__app
8 - approvaldate__app
9 - bitfunded__app
10 - fundeddate__app
11 - dtmapproved__app
12 - dtmdeclined__app
13 - defaultdate__app
14 - chargeoffdate__app
15 - defaultamount__app
16 - chargeoffamount__app
17 - applicationmonth__app
18 - applicationquarter__app
19 - applicationdayofweek__app
20 - bigdealerid__app
21 - bitrolled__app
22 - bigdealertypeid__app
23 - dealerstate__app
24 - bitlhmgroup__app
25 - dealercity__app
26 - dealerzip__app
27 - bitdealerapplicantsamestate__app
28 - bitdealerapplicantsamecity__app
29 - bitdealerapplicantsamezip__app
30 - strdealershiptrackertype__app
31 - inttype__app
32 - fltacquisitionfee__app
33 - fltaddfee__app
34 - fltallowance__app
35 - fltamountfinanced__app
36 - fltapprovedapr_contract__app
37 - fltapproveddebttoincome__app
38 - fltapprovedloantovalue__app
39 - fltapprovedpayment__a

In [9]:
# # app
# list_cols_raw_app_suffix = [col for col in list_cols if '__app' in col]
# # rm ENG
# list_cols_raw_app_suffix = [col for col in list_cols_raw_app_suffix if 'ENG-' not in col]
# # forced feats
# list_cols_force = [
#     'applicationdate__app', 
#     'payment__app', 
#     'amtfinanced__app',
#     'bookvalue__app',
#     'vehicleyear__app', # for vehicle age
#     'dealerstampcreation__app', # for dealership age
#     'intterm__app',
#     'fltapproveddowntotal__app', # for counter offers
# ]
# list_cols_raw_app_suffix = list_cols_raw_app_suffix + list_cols_force
# # rm dups
# list_cols_raw_app_suffix = list(dict.fromkeys(list_cols_raw_app_suffix))

# print(f'There are {len(list_cols_raw_app_suffix)} application features')
# for a, col in enumerate(list_cols_raw_app_suffix):
#     print(f'{a+1} - {col}')

In [10]:
# see nan
ser_isnull = df[list_cols_raw_app_suffix].isnull().mean()
df_isnull_app = pd.DataFrame({
    'feature': list(ser_isnull.index),
    'propna': list(ser_isnull.values),
})
df_isnull_app.sort_values(by='propna', ascending=False, inplace=True)

# save
str_filename = 'df_isnull_app.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_isnull_app.to_csv(str_local_path, index=False)

# show
df_isnull_app

,feature,propna
12,defaultdate__app,1.0
14,defaultamount__app,1.0
65,fltstatesalestax__app,1.0
64,vehiclepricewholesale__app,1.0
63,fltpriceselling__app,1.0
...,...,...
53,bitdealertrack__app,0.0
54,bitrouteone__app,0.0
58,bitmaintenanceagreement__app,0.0
66,uniqueid__app_y,0.0


### Income

In [11]:
list_cols_income = [col for col in df.columns if '__income' in col]
print(f'There are {len(list_cols_income)} income columns')

There are 2 income columns


In [12]:
# see nan
ser_isnull = df[list_cols_income].isnull().mean()
df_isnull_income = pd.DataFrame({
    'feature': list(ser_isnull.index),
    'propna': list(ser_isnull.values),
})
df_isnull_income.sort_values(by='propna', ascending=False, inplace=True)

# save
str_filename = 'df_isnull_income.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_isnull_income.to_csv(str_local_path, index=False)

# show
df_isnull_income

,feature,propna
0,fltgrossmonthly__income_sum,0.001096
1,fltgrossmonthly__income_count,0.001096


### LN

In [13]:
# ln
list_cols_raw_ln_suffix = [col for col in df.columns if '__ln' in col]
print(f'There are {len(list_cols_raw_ln_suffix)} LN features')
for a, col in enumerate(list_cols_raw_ln_suffix):
    print(f'{a+1} - {col}')

There are 242 LN features
1 - uniqueid__ln
2 - biglnriskviewattributesv5id__ln
3 - bigaccountid__ln
4 - bigdebtorid__ln
5 - biglnriskviewscoreid__ln
6 - bitinvalid__ln
7 - dtmstampcreation__ln
8 - attribute_index__ln
9 - inputprovidedfirstname__ln
10 - inputprovidedlastname__ln
11 - inputprovidedstreetaddress__ln
12 - inputprovidedcity__ln
13 - inputprovidedstate__ln
14 - inputprovidedzipcode__ln
15 - inputprovidedssn__ln
16 - inputprovideddateofbirth__ln
17 - inputprovidedphone__ln
18 - inputprovidedlexid__ln
19 - subjectrecordtimeoldest__ln
20 - subjectrecordtimenewest__ln
21 - subjectnewestrecord12month__ln
22 - subjectactivityindex03month__ln
23 - subjectactivityindex06month__ln
24 - subjectactivityindex12month__ln
25 - subjectage__ln
26 - subjectdeceased__ln
27 - subjectssncount__ln
28 - subjectstabilityindex__ln
29 - subjectstabilityprimaryfactor__ln
30 - subjectabilityindex__ln
31 - subjectabilityprimaryfactor__ln
32 - subjectwillingnessindex__ln
33 - subjectwillingnessprimaryfa

In [14]:
# # ln
# list_cols_raw_ln_suffix = [col for col in list_cols if '__ln' in col]
# print(f'There are {len(list_cols_raw_ln_suffix)} LN features')
# for a, col in enumerate(list_cols_raw_ln_suffix):
#     print(f'{a+1} - {col}')

In [15]:
# see nan
ser_isnull = df[list_cols_raw_ln_suffix].isnull().mean()
df_isnull_ln = pd.DataFrame({
    'feature': list(ser_isnull.index),
    'propna': list(ser_isnull.values),
})
df_isnull_ln.sort_values(by='propna', ascending=False, inplace=True)

# save
str_filename = 'df_isnull_ln.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_isnull_ln.to_csv(str_local_path, index=False)

# show
df_isnull_ln

,feature,propna
241,ln_was_empty__ln,1.00000
208,bankcard_reason4__ln,1.00000
217,telecommunications_reason1__ln,1.00000
216,telecommunications_score__ln,1.00000
215,short_term_lending_reason5__ln,1.00000
...,...,...
5,bitinvalid__ln,0.00007
4,biglnriskviewscoreid__ln,0.00007
3,bigdebtorid__ln,0.00007
2,bigaccountid__ln,0.00007


### TU

In [16]:
# tu
list_cols_raw_tu_suffix = [col for col in df.columns if '__tu' in col]
print(f'There are {len(list_cols_raw_tu_suffix)} TU features')

There are 2146 TU features


In [17]:
# # tu
# list_cols_raw_tu_suffix = [col for col in list_cols if '__tu' in col]
# print(f'There are {len(list_cols_raw_tu_suffix)} TU features')

In [18]:
# see nan
ser_isnull = df[list_cols_raw_tu_suffix].isnull().mean()
df_isnull_tu = pd.DataFrame({
    'feature': list(ser_isnull.index),
    'propna': list(ser_isnull.values),
})
df_isnull_tu.sort_values(by='propna', ascending=False, inplace=True)

# save
str_filename = 'df_isnull_tu.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_isnull_tu.to_csv(str_local_path, index=False)

# show
df_isnull_tu

,feature,propna
2145,tu_was_empty__tu,1.000000
1938,au921b__tu,1.000000
1650,p03h__tu,1.000000
1651,linkahit__tu,1.000000
1652,linkbhit__tu,1.000000
...,...,...
703,rvlr73__tu,0.000379
702,rvlr72__tu,0.000379
701,rvlr71__tu,0.000379
700,rvlr70__tu,0.000379
